# 🎯 Week 8: 중단 가능한 문서 처리 파이프라인

## 📚 LLM 성능을 높이는 패턴: 구조화된 출력과 안전한 제어

### 학습 목표
- 작업을 단계별로 나누고 중간 결과 저장
- 언제든 멈추고(STOP) 다시 이어갈 수 있는(resume) 시스템 구축
- 동시 요청을 안전하게 처리하는 대기열 시스템

---

## 📦 Part 0: 환경 설정 및 라이브러리 설치

In [ ]:
# 필수 패키지 설치
!pip install -q langchain langchain-openai

In [ ]:
# 라이브러리 임포트
import os
import json
import pickle
import uuid
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict
import queue
import threading
import time

# LangChain 관련
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

# 경고 메시지 숨기기
import warnings

warnings.filterwarnings("ignore")

print("✅ 모든 라이브러리가 성공적으로 임포트되었습니다!")

✅ 모든 라이브러리가 성공적으로 임포트되었습니다!


## 🔑 Part 1: API 설정 및 기본 구조 정의

In [2]:
from dotenv import load_dotenv

load_dotenv()

# OpenAI API 키 확인
if not os.getenv("OPENAI_API_KEY"):
    print(" * 경고: OPENAI_API_KEY가 .env 파일에 설정되지 않았습니다.")
else:
    # API 키의 처음 10자만 표시 (보안)
    api_key = os.getenv("OPENAI_API_KEY")
    print(f" * OpenAI API 키 로드 완료 !")

 * OpenAI API 키 로드 완료 !


In [ ]:
# LLM 초기화
llm = ChatOpenAI(model="gpt-5-mini")

# 비평용 LLM
critic_llm = ChatOpenAI(model="gpt-5-mini")

print("✅ LLM이 초기화되었습니다!")

✅ LLM이 초기화되었습니다!


In [ ]:
# 데이터 클래스 정의
@dataclass
class StepResult:
    """각 Step의 결과를 저장하는 클래스"""

    step: int
    step_name: str
    result: str
    timestamp: str = None

    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@dataclass
class Checkpoint:
    """체크포인트 정보를 저장하는 클래스"""

    run_id: str
    completed_step: int
    user_goal: str
    intermediate_results: List[str]
    timestamp: str = None

    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@dataclass
class Request:
    """사용자 요청을 저장하는 클래스"""

    id: str
    user_goal: str
    document: str
    status: str = "pending"  # pending, processing, completed, stopped

    def __post_init__(self):
        if self.id is None:
            self.id = f"req_{uuid.uuid4().hex[:8]}"


print("✅ 데이터 클래스 정의 완료!")

✅ 데이터 클래스 정의 완료!


## 🏗️ Part 2: 4단계 파이프라인 구현

작업을 4단계로 나누어 처리합니다:
1. **계획 세우기**: 무엇을 할지 간단한 계획 만들기
2. **입력 읽기**: 문서를 읽고 핵심 후보 뽑기
3. **초안 만들기**: 요약 초안 작성
4. **최종 정리**: 핵심 포인트 + 액션아이템 정리

In [ ]:
class DocumentPipeline:
    """
    문서 처리 파이프라인
    4단계로 나누어 작업을 처리하고 중간 결과를 저장합니다.
    """

    def __init__(self):
        self.llm = llm
        self.stop_flag = False
        self.current_run_id = None
        self.checkpoint = None

    def step1_plan(self, user_goal: str, document: str) -> StepResult:
        """Step 1: 계획 세우기"""
        print("\n🔍 Step 1: 계획 세우기...")

        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "당신은 문서 분석 전문가입니다. 사용자의 요청을 분석하여 작업 계획을 세워주세요.",
                ),
                (
                    "human",
                    """
            사용자 요청: {user_goal}
            문서 길이: {doc_length}자
            
            이 작업을 수행하기 위한 간단한 계획을 3-4줄로 작성해주세요.
            """,
                ),
            ]
        )

        response = self.llm.invoke(
            prompt.format_messages(user_goal=user_goal, doc_length=len(document))
        )

        return StepResult(step=1, step_name="계획 세우기", result=response.content)

    def step2_read(self, document: str, plan: str) -> StepResult:
        """Step 2: 입력 읽기"""
        print("\n📖 Step 2: 입력 읽기...")

        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", "문서에서 핵심 정보를 추출하는 전문가입니다."),
                (
                    "human",
                    """
            계획: {plan}
            
            문서:
            {document}
            
            위 문서에서 핵심 포인트 5개를 추출해주세요.
            """,
                ),
            ]
        )

        response = self.llm.invoke(
            prompt.format_messages(
                plan=plan, document=document[:2000]  # 문서가 너무 길면 앞부분만
            )
        )

        return StepResult(step=2, step_name="입력 읽기", result=response.content)

    def step3_draft(self, key_points: str, user_goal: str) -> StepResult:
        """Step 3: 초안 만들기"""
        print("\n✍️ Step 3: 초안 만들기...")

        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", "요약 초안을 작성하는 전문가입니다."),
                (
                    "human",
                    """
            사용자 요청: {user_goal}
            핵심 포인트:
            {key_points}
            
            위 정보를 바탕으로 200자 내외의 요약 초안을 작성해주세요.
            """,
                ),
            ]
        )

        response = self.llm.invoke(
            prompt.format_messages(user_goal=user_goal, key_points=key_points)
        )

        return StepResult(step=3, step_name="초안 만들기", result=response.content)

    def step4_finalize(self, draft: str, key_points: str) -> StepResult:
        """Step 4: 최종 정리"""
        print("\n🎯 Step 4: 최종 정리...")

        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", "최종 결과를 정리하는 전문가입니다."),
                (
                    "human",
                    """
            초안: {draft}
            핵심 포인트: {key_points}
            
            최종 결과를 다음 형식으로 정리해주세요:
            
            [요약]
            (여기에 최종 요약)
            
            [핵심 포인트]
            • 포인트 1
            • 포인트 2
            • 포인트 3
            
            [액션 아이템]
            1. (담당자) - (할 일) - (기한)
            2. (담당자) - (할 일) - (기한)
            """,
                ),
            ]
        )

        response = self.llm.invoke(
            prompt.format_messages(draft=draft, key_points=key_points)
        )

        return StepResult(step=4, step_name="최종 정리", result=response.content)


print("✅ DocumentPipeline 클래스 정의 완료!")

✅ DocumentPipeline 클래스 정의 완료!


## 💾 Part 3: 중간출력 및 체크포인트 관리

In [ ]:
class FileManager:
    """
    파일 저장 및 체크포인트 관리 클래스
    """

    def __init__(self, base_dir: str = "runs"):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(exist_ok=True)

    def save_step_result(self, run_id: str, step_result: StepResult):
        """Step 결과를 파일로 저장"""
        run_dir = self.base_dir / run_id
        run_dir.mkdir(exist_ok=True)

        filename = f"step_{step_result.step:02d}.json"
        filepath = run_dir / filename

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(asdict(step_result), f, ensure_ascii=False, indent=2)

        print(f"  💾 저장됨: {filepath}")
        return str(filename)

    def save_checkpoint(self, run_id: str, checkpoint: Checkpoint):
        """체크포인트 저장"""
        checkpoint_dir = self.base_dir / "checkpoints"
        checkpoint_dir.mkdir(exist_ok=True)

        filepath = checkpoint_dir / f"{run_id}_checkpoint.json"

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(asdict(checkpoint), f, ensure_ascii=False, indent=2)

        print(f"  🔖 체크포인트 저장됨: {filepath}")
        return str(filepath)

    def load_checkpoint(self, run_id: str) -> Optional[Checkpoint]:
        """체크포인트 불러오기"""
        checkpoint_path = self.base_dir / "checkpoints" / f"{run_id}_checkpoint.json"

        if not checkpoint_path.exists():
            print(f"  ❌ 체크포인트를 찾을 수 없습니다: {run_id}")
            return None

        with open(checkpoint_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        checkpoint = Checkpoint(**data)
        print(f"  ✅ 체크포인트 로드됨: Step {checkpoint.completed_step}까지 완료")
        return checkpoint

    def load_step_result(self, run_id: str, step: int) -> Optional[StepResult]:
        """특정 Step 결과 불러오기"""
        filepath = self.base_dir / run_id / f"step_{step:02d}.json"

        if not filepath.exists():
            return None

        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        return StepResult(**data)

    def save_final_result(self, run_id: str, result: str):
        """최종 결과 저장"""
        run_dir = self.base_dir / run_id
        filepath = run_dir / "final_output.txt"

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(result)

        print(f"  📄 최종 결과 저장됨: {filepath}")


# 파일 매니저 인스턴스 생성
file_manager = FileManager()
print("✅ FileManager 준비 완료!")

✅ FileManager 준비 완료!


## 🔄 Part 4: STOP/Resume 기능 구현

In [ ]:
class PipelineController:
    """
    파이프라인을 제어하는 컨트롤러
    STOP과 Resume 기능을 제공합니다.
    """

    def __init__(self):
        self.pipeline = DocumentPipeline()
        self.file_manager = FileManager()
        self.stop_requested = False
        self.current_run_id = None

    def run(self, request: Request, resume_from: Optional[str] = None) -> str:
        """
        파이프라인 실행
        resume_from: 재개할 run_id (None이면 처음부터)
        """
        # Run ID 설정
        if resume_from:
            self.current_run_id = resume_from
            checkpoint = self.file_manager.load_checkpoint(resume_from)
            if not checkpoint:
                return "체크포인트를 찾을 수 없습니다."

            start_step = checkpoint.completed_step + 1
            print(f"\n🔄 Resume: Step {start_step}부터 이어서 실행합니다.")
        else:
            self.current_run_id = f"run_{uuid.uuid4().hex[:8]}"
            start_step = 1
            print(f"\n🚀 새로운 실행 시작: {self.current_run_id}")

        # 이전 결과 불러오기 (resume인 경우)
        step_results = {}
        if resume_from:
            for i in range(1, start_step):
                result = self.file_manager.load_step_result(resume_from, i)
                if result:
                    step_results[i] = result

        # 파이프라인 실행
        self.stop_requested = False

        try:
            # Step 1: 계획 세우기
            if start_step <= 1:
                if self.check_stop():
                    return self.handle_stop(request, 0)

                result1 = self.pipeline.step1_plan(request.user_goal, request.document)
                step_results[1] = result1
                self.file_manager.save_step_result(self.current_run_id, result1)

                if self.check_stop():
                    return self.handle_stop(request, 1)

            # Step 2: 입력 읽기
            if start_step <= 2:
                plan = step_results.get(1).result if 1 in step_results else ""
                result2 = self.pipeline.step2_read(request.document, plan)
                step_results[2] = result2
                self.file_manager.save_step_result(self.current_run_id, result2)

                if self.check_stop():
                    return self.handle_stop(request, 2)

            # Step 3: 초안 만들기
            if start_step <= 3:
                key_points = step_results.get(2).result if 2 in step_results else ""
                result3 = self.pipeline.step3_draft(key_points, request.user_goal)
                step_results[3] = result3
                self.file_manager.save_step_result(self.current_run_id, result3)

                if self.check_stop():
                    return self.handle_stop(request, 3)

            # Step 4: 최종 정리
            if start_step <= 4:
                draft = step_results.get(3).result if 3 in step_results else ""
                key_points = step_results.get(2).result if 2 in step_results else ""
                result4 = self.pipeline.step4_finalize(draft, key_points)
                step_results[4] = result4
                self.file_manager.save_step_result(self.current_run_id, result4)

            # 최종 결과 저장
            final_output = step_results[4].result
            self.file_manager.save_final_result(self.current_run_id, final_output)

            print("\n✅ 파이프라인 완료!")
            return final_output

        except Exception as e:
            print(f"\n❌ 오류 발생: {e}")
            return str(e)

    def stop(self):
        """실행 중단 요청"""
        self.stop_requested = True
        print("\n⚠️ STOP 요청됨 - 현재 Step 완료 후 중단됩니다...")

    def check_stop(self) -> bool:
        """중단 요청 확인"""
        return self.stop_requested

    def handle_stop(self, request: Request, completed_step: int) -> str:
        """중단 처리 및 체크포인트 저장"""
        print(f"\n🛑 Step {completed_step} 완료 후 중단됨")

        # 체크포인트 생성
        intermediate_results = []
        for i in range(1, completed_step + 1):
            intermediate_results.append(f"step_{i:02d}.json")

        checkpoint = Checkpoint(
            run_id=self.current_run_id,
            completed_step=completed_step,
            user_goal=request.user_goal,
            intermediate_results=intermediate_results,
        )

        self.file_manager.save_checkpoint(self.current_run_id, checkpoint)

        return f"중단됨. resume('{self.current_run_id}')로 이어서 실행 가능합니다."


# 컨트롤러 인스턴스 생성
controller = PipelineController()
print("✅ PipelineController 준비 완료!")

✅ PipelineController 준비 완료!


## 📋 Part 5: 대기열(Queue) 시스템 구현

In [ ]:
class RequestQueue:
    """
    요청 대기열 관리 시스템
    FIFO 방식으로 동시 요청을 순차 처리합니다.
    """

    def __init__(self):
        self.queue = queue.Queue()
        self.is_processing = False
        self.current_request = None
        self.controller = PipelineController()

    def add_request(self, request: Request):
        """요청을 대기열에 추가"""
        self.queue.put(request)
        queue_size = self.queue.qsize()

        if self.is_processing:
            print(f"[대기 중] {queue_size}개 요청이 대기 중입니다")
        else:
            self.process_next()

    def process_next(self):
        """다음 요청 처리"""
        if self.is_processing or self.queue.empty():
            return

        self.is_processing = True
        self.current_request = self.queue.get()

        print(
            f"\n[실행 중] 요청 #{self.current_request.id}: {self.current_request.user_goal}"
        )
        print(f"[대기 중] {self.queue.qsize()}개 요청이 대기 중입니다")

        # 실제 처리 (별도 스레드에서 실행하면 더 좋음)
        result = self.controller.run(self.current_request)

        print(f"\n[완료] 요청 #{self.current_request.id} 처리 완료")

        self.is_processing = False
        self.current_request = None

        # 다음 요청 처리
        if not self.queue.empty():
            self.process_next()

    def get_status(self) -> Dict:
        """현재 상태 조회"""
        return {
            "is_processing": self.is_processing,
            "current_request": (
                self.current_request.id if self.current_request else None
            ),
            "queue_size": self.queue.qsize(),
        }


# 대기열 시스템 인스턴스 생성
request_queue = RequestQueue()
print("✅ RequestQueue 준비 완료!")

✅ RequestQueue 준비 완료!


## 🧪 Part 6: 테스트 시나리오

### 준비: 샘플 문서

In [9]:
# 테스트용 샘플 문서
sample_document = """
인공지능(AI) 기술의 발전과 기업 적용 전략

최근 인공지능 기술이 빠르게 발전하면서 많은 기업들이 AI를 도입하고 있습니다.
특히 대규모 언어모델(LLM)의 등장으로 자연어 처리 분야에서 혁신적인 변화가 일어나고 있습니다.

1. 현재 AI 기술 동향
- ChatGPT와 같은 대화형 AI의 보편화
- 이미지 생성 AI의 창작 분야 진출
- 자동화 및 최적화 도구로서의 AI 활용 증가

2. 기업 도입 시 고려사항
- 데이터 보안 및 개인정보 보호
- 기존 시스템과의 통합
- 직원 교육 및 변화 관리
- ROI 측정 및 성과 평가 체계 구축

3. 향후 전망
AI 기술은 계속 발전할 것이며, 기업들은 이를 전략적으로 활용하여
경쟁 우위를 확보해야 합니다. 특히 고객 서비스, 제품 개발, 운영 효율화
분야에서 AI의 역할이 더욱 중요해질 것으로 예상됩니다.
"""

print("✅ 샘플 문서 준비 완료!")
print(f"문서 길이: {len(sample_document)}자")

✅ 샘플 문서 준비 완료!
문서 길이: 430자


### 📝 시나리오 A: 정상 실행

In [ ]:
print("=" * 60)
print("📝 시나리오 A: 정상 실행")
print("=" * 60)

# 요청 생성
request_a = Request(
    id="test_a",
    user_goal="이 문서를 요약하고 주요 액션아이템을 추출해주세요",
    document=sample_document,
)

# 파이프라인 실행
controller_a = PipelineController()
result_a = controller_a.run(request_a)

print("\n" + "=" * 60)
print("최종 결과:")
print("=" * 60)
print(result_a[:500])  # 결과 일부만 표시

📝 시나리오 A: 정상 실행

🚀 새로운 실행 시작: run_c383c514

🔍 Step 1: 계획 세우기...
  💾 저장됨: runs\run_c383c514\step_01.json

📖 Step 2: 입력 읽기...
  💾 저장됨: runs\run_c383c514\step_02.json

✍️ Step 3: 초안 만들기...
  💾 저장됨: runs\run_c383c514\step_03.json

🎯 Step 4: 최종 정리...
  💾 저장됨: runs\run_c383c514\step_04.json
  📄 최종 결과 저장됨: runs\run_c383c514\final_output.txt

✅ 파이프라인 완료!

최종 결과:
[요약]
대규모 언어모델(LLM)을 중심으로 대화형 AI, 이미지 생성, 자동화 도구가 자연어처리 분야를 빠르게 혁신하고 있다. 기업은 보안·통합·교육·ROI 문제를 사전 해결하면서 고객서비스, 제품개발, 운영 등 우선 적용 영역에서 파일럿을 통해 역량을 확립해야 경쟁 우위를 확보할 수 있다.

[핵심 포인트]
• 기술 트렌드: LLM 기반 대화형 AI의 보편화, 이미지 생성의 창작·마케팅 활용 확대, 자동화·최적화 도구의 업무 효율화 기여.  
• 도입 고려사항: 데이터 보안·개인정보 보호, 기존 시스템과의 기술·운영 통합, 직원 교육·변화관리, 명확한 ROI·성과지표 설정 필요.  
• 실행 전략: 고객서비스·제품개발·운영을 우선 적용 대상으로 삼아 파일럿 → 검증 → 확장 방식으로 단계적 도입하고 지속적 기술 모니터링으로 경쟁력 유지.

[액션 아이템]
1. (CTO) - AI 인프라·데이터 보안 현황 점검 및 보안·거버넌스 가이드라인 수립 - 4주  
2. (CPO/Head 


### 🛑 시나리오 B: STOP 후 재개

주의: 실제 환경에서는 별도 스레드나 비동기로 구현해야 하지만, 노트북에서는 시뮬레이션으로 진행합니다.

In [ ]:
print("=" * 60)
print("🛑 시나리오 B: STOP 후 재개")
print("=" * 60)

# 요청 생성
request_b = Request(
    id="test_b",
    user_goal="AI 도입 전략에 대한 상세 분석을 작성해주세요",
    document=sample_document,
)


# Step 2 이후 중단 시뮬레이션
class StoppablePipelineController(PipelineController):
    def __init__(self, stop_after_step=2):
        super().__init__()
        self.stop_after_step = stop_after_step
        self.step_count = 0

    def check_stop(self):
        self.step_count += 1
        if self.step_count > self.stop_after_step:
            return True
        return False


# Step 2까지만 실행하고 중단
controller_b = StoppablePipelineController(stop_after_step=2)
result_b1 = controller_b.run(request_b)
print(f"\n결과: {result_b1}")

# 중단된 run_id 추출
stopped_run_id = controller_b.current_run_id
print(f"\n중단된 Run ID: {stopped_run_id}")

🛑 시나리오 B: STOP 후 재개

🚀 새로운 실행 시작: run_feb7b161

🔍 Step 1: 계획 세우기...
  💾 저장됨: runs\run_feb7b161\step_01.json

📖 Step 2: 입력 읽기...
  💾 저장됨: runs\run_feb7b161\step_02.json

🛑 Step 2 완료 후 중단됨
  🔖 체크포인트 저장됨: runs\checkpoints\run_feb7b161_checkpoint.json

결과: 중단됨. resume('run_feb7b161')로 이어서 실행 가능합니다.

중단된 Run ID: run_feb7b161


In [ ]:
# Resume 실행
print("\n" + "=" * 60)
print("🔄 Resume 실행")
print("=" * 60)

# 새로운 컨트롤러로 이어서 실행
controller_b_resume = PipelineController()
result_b2 = controller_b_resume.run(request_b, resume_from=stopped_run_id)

print("\n" + "=" * 60)
print("Resume 후 최종 결과:")
print("=" * 60)
print(result_b2[:500])


🔄 Resume 실행
  ✅ 체크포인트 로드됨: Step 2까지 완료

🔄 Resume: Step 3부터 이어서 실행합니다.

✍️ Step 3: 초안 만들기...
  💾 저장됨: runs\run_feb7b161\step_03.json

🎯 Step 4: 최종 정리...
  💾 저장됨: runs\run_feb7b161\step_04.json
  📄 최종 결과 저장됨: runs\run_feb7b161\final_output.txt

✅ 파이프라인 완료!

Resume 후 최종 결과:
[요약]
LLM 중심의 AI 발전으로 기업 도입이 가속화되고 있으며, 특히 대화형 AI, 이미지 생성, 자동화·최적화 도구가 확산되고 있다. 도입 시 데이터 보안·개인정보 보호가 주요 리스크로 작용하고, 기존 시스템 통합·직원 교육·변화관리 등 조직적·기술적 준비가 필요하다. 고객 서비스, 제품 개발, 운영 효율화에서 경쟁우위를 얻기 위해 ROI 및 성과평가 체계를 구축해 전략적으로 활용해야 한다.

[핵심 포인트]
• LLM(대규모 언어모델) 중심의 자연어처리 혁신이 기업 내 AI 도입을 촉진하며 대화형·이미지 생성·자동화 기술이 빠르게 확산되고 있다.  
• 도입 리스크로는 데이터 보안·개인정보 보호가 가장 중요하며, 기존 시스템 통합·직원 교육·변화관리 등 조직적 준비가 필수적이다.  
• AI는 고객 서비스·제품 개발·운영 효율화로 경쟁우위를 제공하므로 ROI 및 성과평가 체계를 마련해 우선순위를 정하고 전략적으로 투자해야 한다.

[액션 아이템]
1. (CTO / AI팀장)


### 📋 시나리오 C: 대기열 테스트

In [ ]:
print("=" * 60)
print("📋 시나리오 C: 대기열 테스트")
print("=" * 60)


# 간단한 대기열 시뮬레이션을 위한 Mock 파이프라인
class MockPipelineController:
    """테스트용 빠른 실행 파이프라인"""

    def run(self, request: Request, resume_from=None) -> str:
        print(f"  처리 시작: {request.id}")
        time.sleep(2)  # 2초 대기 (실제 처리 시뮬레이션)
        print(f"  처리 완료: {request.id}")
        return f"결과: {request.id} 완료"


# Mock 컨트롤러를 사용하는 대기열
class TestRequestQueue(RequestQueue):
    def __init__(self):
        super().__init__()
        self.controller = MockPipelineController()


# 테스트용 대기열 생성
test_queue = TestRequestQueue()

# 요청 2개 연속 추가
request_c1 = Request(
    id="req_001", user_goal="첫 번째 요청: 문서 요약", document="문서 내용 1"
)

request_c2 = Request(
    id="req_002", user_goal="두 번째 요청: 액션아이템 추출", document="문서 내용 2"
)

print("\n요청 2개를 연속으로 추가합니다...\n")
test_queue.add_request(request_c1)
time.sleep(0.5)  # 약간의 지연
test_queue.add_request(request_c2)

print("\n현재 상태:", test_queue.get_status())

📋 시나리오 C: 대기열 테스트

요청 2개를 연속으로 추가합니다...


[실행 중] 요청 #req_001: 첫 번째 요청: 문서 요약
[대기 중] 0개 요청이 대기 중입니다
  처리 시작: req_001
  처리 완료: req_001

[완료] 요청 #req_001 처리 완료

[실행 중] 요청 #req_002: 두 번째 요청: 액션아이템 추출
[대기 중] 0개 요청이 대기 중입니다
  처리 시작: req_002
  처리 완료: req_002

[완료] 요청 #req_002 처리 완료

현재 상태: {'is_processing': False, 'current_request': None, 'queue_size': 0}


## 📊 Part 7: 결과 확인 및 파일 구조 보기

In [ ]:
import os


def show_directory_structure(path, prefix="", max_depth=3, current_depth=0):
    """디렉토리 구조를 트리 형태로 표시"""
    if current_depth > max_depth:
        return

    items = sorted(os.listdir(path))

    for i, item in enumerate(items):
        item_path = os.path.join(path, item)
        is_last = i == len(items) - 1

        # 현재 아이템 출력
        current_prefix = "└── " if is_last else "├── "
        print(f"{prefix}{current_prefix}{item}")

        # 디렉토리면 재귀 호출
        if os.path.isdir(item_path):
            next_prefix = prefix + ("    " if is_last else "│   ")
            show_directory_structure(
                item_path, next_prefix, max_depth, current_depth + 1
            )


print("📁 생성된 파일 구조:")
print("\nruns/")
show_directory_structure("runs")

📁 생성된 파일 구조:

runs/
├── checkpoints
│   └── run_feb7b161_checkpoint.json
├── run_c383c514
│   ├── final_output.txt
│   ├── step_01.json
│   ├── step_02.json
│   ├── step_03.json
│   └── step_04.json
└── run_feb7b161
    ├── final_output.txt
    ├── step_01.json
    ├── step_02.json
    ├── step_03.json
    └── step_04.json


In [ ]:
# 특정 Step 결과 내용 확인
def show_step_content(run_id: str, step: int):
    """특정 Step의 저장된 내용 확인"""
    filepath = Path("runs") / run_id / f"step_{step:02d}.json"

    if filepath.exists():
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(f"\n📄 {run_id}/step_{step:02d}.json 내용:")
        print("=" * 60)
        print(f"Step: {data['step']}")
        print(f"Step Name: {data['step_name']}")
        print(f"Timestamp: {data['timestamp']}")
        print(f"Result (첫 200자): {data['result'][:200]}...")
    else:
        print(f"파일을 찾을 수 없습니다: {filepath}")


# 예시: 첫 번째 테스트의 Step 1 결과 확인
if "controller_a" in locals():
    show_step_content(controller_a.current_run_id, 1)


📄 run_c383c514/step_01.json 내용:
Step: 1
Step Name: 계획 세우기
Timestamp: 2026-01-05 19:05:09
Result (첫 200자): 1. 문서를 전체적으로 읽어 핵심 주제와 목적을 파악합니다.
2. 핵심 문장과 키워드를 추출해 논점을 압축하여 2–3문장 내로 요약합니다.
3. 문서에서 명시적·암시적 행동 항목을 식별하고 책임자·기한·우선순위를 가능한 한 명확히 표기합니다.
4. 요약문과 액션아이템 목록을 검토·정리해 최종 결과물로 제공합니다....
